In [1]:
import os
from openai import OpenAI
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

f:\gits\suresh-putla\ai-shopping-assistant-lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [30]:
ls_client = Client()

In [31]:
dataset_name = "Amazon-shopping-collection-01-dataset"
dataset = ls_client.read_dataset(
    dataset_name=dataset_name
)

In [32]:
dataset

Dataset(name='Amazon-shopping-collection-01-dataset', description='Dataset for Amazon-shopping-collection-01', data_type=<DataType.kv: 'kv'>, id=UUID('fc7c8b9f-6d39-49f8-84ae-358037dd5795'), created_at=datetime.datetime(2026, 6, 28, 13, 8, 58, 554466, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 6, 28, 13, 8, 58, 554466, tzinfo=TzInfo(0)), example_count=32, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'Windows-11-10.0.26200-SP0', 'sdk_version': '0.9.3', 'runtime_version': '3.13.5', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [33]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

{'ground_truth': 'You have at least two binocular options for bird watching. One is a 12x42 model with BAK4 prism, FMC lenses, low night vision, and IPX7 waterproofing. Another is the LUXUN 10x50 model with low light night vision, BAK4 prism, FMC optics, and IPX3 waterproof/fog-proof protection. The main differences are magnification, lens size, and waterproof rating.',
 'reference_context_ids': ['B09KH916K5', 'B08JPK7XR4'],
 'reference_descriptions': ['12x42 Compact Binoculars with Low Night Vision for Adults Kids Waterproof BAK4 Prism FMC Lens Binoculars for Hunting Bird Wildlife Watching Travel Sports✨【High Power Binoculars】 - 12x optics magnification binoculars with 42mm big objective lens, large field of view and long distance, see farther and wider. Perfect binoculars for courtyard birding, hunting, hiking, camping, climbing, safari, sports games and concerts. This is ideal binoculars for adults and teens. ✨【Professional Binoculars】 - Professional FMC Multi-Coated Lens, light tra

In [34]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs

{'question': 'What binocular options do you have for bird watching, and how do they differ?'}

In [35]:

reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

### RAG Pipeline

In [2]:
model = "text-embedding-3-small"
collection_name="Amazon-shopping-collection-01"
qdrant_client = QdrantClient(url="http://localhost:6333")
model_name = "gpt-5.4-nano"
openai = OpenAI()

In [3]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding

In [21]:
def retrieve_context(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(collection_name=collection_name, query=query_embedding, limit=k)
    context_ids = []
    context = []
    scores = []
    ratings = []

    print(results)
    for pt in results.points:
        context_ids.append(pt.payload['parent_asin'])
        context.append(pt.payload['preprocessed_description'])
        scores.append(pt.score)
        ratings.append(pt.payload['average_rating'])
        
    return {
        'retrieved_context_ids': context_ids,
        'retrieved_context': context,
        'similarity_scores': scores,
        'retrieved_context_ratings': ratings
    }

In [13]:
def process_context(context):
    formated_context = ""
    for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrieved_context'], context['retrieved_context_ratings']):
        formated_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formated_context

In [14]:
def build_prompt(context, query):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - answer the question based on the provided context only.
    - never use word context and refer to it as the available products.
    - do not use markdown formatting.

    Context:
    {context}

    Question:
    {query}
    """

    return prompt

In [15]:
def generate_answer(prompt):
    
    response = openai.chat.completions.create(
            model=model_name,
            messages=[{"role":"system","content": prompt}]
                     )
    
    return response.choices[0].message.content

In [22]:
def rag_pipeline(query, top_k= 5):
    retrieved_context = retrieve_context(query, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    llm_answer = generate_answer(prompt)
    
    final_answer= {
        "query": query,
        "answer": llm_answer,
        "retrieved_context_ids": retrieved_context['retrieved_context_ids'],
        "retrieved_context": retrieved_context['retrieved_context'],
    }
    return final_answer

In [25]:
rag_pipeline("do you have samsung phones ?")

points=[ScoredPoint(id=33, version=1, score=0.33944556, payload={'preprocessed_description': 'TJS Galaxy J3 Emerge/J3 Prime/Amp Prime 2/Express Prime 2/Sol 2/J3 Mission/J3 Luna Pro/J3 Eclipse Case, with [Tempered Glass Screen Protector] Shock Absorbing Kickstand Silicone Inner Layer (Black)', 'image': 'https://m.media-amazon.com/images/I/510VEheOP1L._AC_.jpg', 'rating_number': 240, 'price': None, 'average_rating': 4.3, 'parent_asin': 'B01MYDWWDA'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=18, version=1, score=0.33071733, payload={'preprocessed_description': 'CASZONE Case for Samsung Galaxy Tab S8 2022/Tab S7 2020 11 inch, Model SM-X700/X706/T870/T875/T878 Tablet Cover with Screen Protector Pencil Holder 360 Degree Rotating Kickstand,BlackCompatible Model: This case is specially designed for galaxy Tab S8 2022/Tab S7 2020 11 inch (SM-X700/X706/T870/T875/T878). Please confirm the model before purchase. For long-distance transportation, please feel free to contact us

{'query': 'do you have samsung phones ?',
 'answer': 'Yes, we have cases for Samsung devices, but not for Samsung phones specifically in the available products list.\n\nAvailable Samsung items:\n- Samsung Galaxy Tab S8 (2022) / Tab S7 (2020) 11-inch case with screen protector and 360° kickstand (with S-Pen holder; S-Pen not included)\n- Samsung Tab S3 cover (magnetic connection, slim/lightweight, white)\n\nIf you tell me your exact Samsung phone model (e.g., Galaxy A14, S23, etc.), I can check whether we have a matching case.',
 'retrieved_context_ids': ['B01MYDWWDA',
  'B0936SY7XW',
  'B073S96GCL',
  'B09FSZ5T32',
  'B07H6Z1RW7'],
 'retrieved_context': ['TJS Galaxy J3 Emerge/J3 Prime/Amp Prime 2/Express Prime 2/Sol 2/J3 Mission/J3 Luna Pro/J3 Eclipse Case, with [Tempered Glass Screen Protector] Shock Absorbing Kickstand Silicone Inner Layer (Black)',
  'CASZONE Case for Samsung Galaxy Tab S8 2022/Tab S7 2020 11 inch, Model SM-X700/X706/T870/T875/T878 Tablet Cover with Screen Protector

### RAGAS Metrics

In [26]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

C:\Users\sures\AppData\Local\Temp\ipykernel_6904\3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
C:\Users\sures\AppData\Local\Temp\ipykernel_6904\3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
C:\Users\sures\AppData\Local\Temp\ipykernel_6904\3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'r

In [27]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

C:\Users\sures\AppData\Local\Temp\ipykernel_6904\840510326.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
C:\Users\sures\AppData\Local\Temp\ipykernel_6904\840510326.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [36]:
reference_input

{'question': 'What binocular options do you have for bird watching, and how do they differ?'}

In [37]:
reference_output

{'ground_truth': 'You have at least two binocular options for bird watching. One is a 12x42 model with BAK4 prism, FMC lenses, low night vision, and IPX7 waterproofing. Another is the LUXUN 10x50 model with low light night vision, BAK4 prism, FMC optics, and IPX3 waterproof/fog-proof protection. The main differences are magnification, lens size, and waterproof rating.',
 'reference_context_ids': ['B09KH916K5', 'B08JPK7XR4'],
 'reference_descriptions': ['12x42 Compact Binoculars with Low Night Vision for Adults Kids Waterproof BAK4 Prism FMC Lens Binoculars for Hunting Bird Wildlife Watching Travel Sports✨【High Power Binoculars】 - 12x optics magnification binoculars with 42mm big objective lens, large field of view and long distance, see farther and wider. Perfect binoculars for courtyard birding, hunting, hiking, camping, climbing, safari, sports games and concerts. This is ideal binoculars for adults and teens. ✨【Professional Binoculars】 - Professional FMC Multi-Coated Lens, light tra

In [38]:
result = rag_pipeline(reference_input["question"])

points=[ScoredPoint(id=12, version=1, score=0.51213896, payload={'preprocessed_description': '12x42 Compact Binoculars with Low Night Vision for Adults Kids Waterproof BAK4 Prism FMC Lens Binoculars for Hunting Bird Wildlife Watching Travel Sports✨【High Power Binoculars】 - 12x optics magnification binoculars with 42mm big objective lens, large field of view and long distance, see farther and wider. Perfect binoculars for courtyard birding, hunting, hiking, camping, climbing, safari, sports games and concerts. This is ideal binoculars for adults and teens. ✨【Professional Binoculars】 - Professional FMC Multi-Coated Lens, light transmittance up to 98.5%. Reduce chromatic aberration and Images blurry, provide crystal clear and sharp images. Professional BAK-4 prism(BAK 4 has superior reflective and dispersion qualities compared to BAK 7) ensure higher refractive index, more transparent imaging, giving you full HD VR vision experience. ✨【Perfect Eyepiece Size】 - 20MM eyepiece size is perfec

In [39]:
result

{'query': 'What binocular options do you have for bird watching, and how do they differ?',
 'answer': 'You have two binocular options in the available products that are suitable for bird watching:\n\n1) B09KH916K5 (12x42 compact binoculars)\n- Magnification/objective: 12x42\n- Lens/coatings: FMC multi-coated lens, BAK4 prism\n- Low-light/night vision: described as “low night vision”\n- Waterproof: IPX7 waterproof rating (water spray resistant for up to at least 3 minutes)\n- Other notes: compact design; easy central focus; 20mm eyepiece size\n\n2) B08JPK7XR4 (LUXUN 10x50 binoculars)\n- Magnification/objective: 10x50\n- Lens/coatings: FMC + BAK4 prism, light transmittance up to 99.58%\n- Low-light/night vision: described as “low light night vision”\n- Waterproof: IPX3 waterproof and fog-proof (not meant to be immersed)\n- Other notes: larger 50mm objective and 28mm eyepiece for more light gathering; includes lens caps, cloth, bag, and strap; 10x magnification with focus wheels\n\nHow th

In [47]:
async def ragas_context_precision_id_based(run, example):

    print(f"retrieved_context_ids: {run["retrieved_context_ids"]}")
    print(f"reference_context_ids: {example["reference_context_ids"]}")
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )
    #print(sample)
    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [52]:
print(await ragas_context_precision_id_based(result, reference_output))

retrieved_context_ids: ['B09KH916K5', 'B08JPK7XR4', 'B0C3YKQT3L', 'B08N558522', 'B01CYAZKAI']
reference_context_ids: ['B09KH916K5', 'B08JPK7XR4']
0.4


In [54]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [55]:
await ragas_context_recall_id_based(result, reference_output)

1.0

In [72]:
async def ragas_faithfulness(run, example):
    print("----------------")
    print(f"Query:{run["query"]}")
    print("----------------")
    print(f"answer:{run["answer"]}")
    print("----------------")
    print(f"retrieved_contexts:{run["retrieved_context"]}")
    print("----------------")
    for rc in run["retrieved_context"]:
        print(f"RC:{rc}")


    sample = SingleTurnSample(
            user_input=run["query"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)

In [73]:
await ragas_faithfulness(result, reference_output)

----------------
Query:What binocular options do you have for bird watching, and how do they differ?
----------------
answer:You have two binocular options in the available products that are suitable for bird watching:

1) B09KH916K5 (12x42 compact binoculars)
- Magnification/objective: 12x42
- Lens/coatings: FMC multi-coated lens, BAK4 prism
- Low-light/night vision: described as “low night vision”
- Waterproof: IPX7 waterproof rating (water spray resistant for up to at least 3 minutes)
- Other notes: compact design; easy central focus; 20mm eyepiece size

2) B08JPK7XR4 (LUXUN 10x50 binoculars)
- Magnification/objective: 10x50
- Lens/coatings: FMC + BAK4 prism, light transmittance up to 99.58%
- Low-light/night vision: described as “low light night vision”
- Waterproof: IPX3 waterproof and fog-proof (not meant to be immersed)
- Other notes: larger 50mm objective and 28mm eyepiece for more light gathering; includes lens caps, cloth, bag, and strap; 10x magnification with focus wheels



0.9285714285714286

In [77]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
        user_input=run["query"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [78]:
await ragas_relevancy(result, reference_output)

np.float64(0.9158765034765682)